# Aid Activity

**Project:** Pali Watch

**Notebook:** `aid_activity.ipynb`

**Author:** Yuseof J

**Created:** 11/11/2026 | **Last updated:** 14/11/2026

## Purpose
This file maps the more granular components of aid activity and coverage in Palestine. It dives into actual timeframes of programs as well as specific activity of aid programs.

## Inputs

- `data/raw/boundaries/`: HDX admin boundaries (COD-AB)

## Outputs
- `data/processed/[filename].csv`: what it contains

## Notes and caveats
- Data limitations: iati data lacks location for 30-60% of activities. Additionally, it only represents aid activity that is reported, it is not an exhaustive list of aid operations
- as of 11/14/2026, setting this component of the project aside to progress with stablishing the rest of the project pipeline

In [40]:
import datetime
import numpy as np
import pandas as pd
import geopandas as gpd
from pathlib import Path
import matplotlib.pyplot as plt

In [41]:
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
    
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

CRS = 'EPSG:4326'

### Data imports / exports

In [42]:
# import paths
PATH_TO_ADMIN_BOUNDARIES = f'{RAW}/pse_admin_boundaries.shp/pse_admin2.shp'
PATH_TO_AID_ACTIVITY_GEO = f'{RAW}/iati-activity-locations-in-state-of-palestine.csv'
PATH_TO_AID_ACTIVITY_DETAILS = f'{RAW}/iati-activities-in-state-of-palestine-no-location-information.csv'

# export paths
PATH_TO_GEOLOCATED_EVENTS_GPKG = f'{PROCESSED}/acled/conflict_events_geolocated.gpkg'
PATH_TO_GEOLOCATED_EVENTS_DF = f'{PROCESSED}/acled/conflict_events_geolocated.csv'

# import data
df_aid_activity_loc = pd.read_csv(PATH_TO_AID_ACTIVITY_GEO)
df_aid_activity_details = pd.read_csv(PATH_TO_AID_ACTIVITY_DETAILS)
gdf_boundaries = gpd.read_file(PATH_TO_ADMIN_BOUNDARIES)

In [43]:
df_aid_activity_details.head()

,aid,reporting,reporting_ref,funder_ref,title,slug,status_code,day_start,day_end,day_length,...,commitment_gbp,spend_gbp,commitment_cad,spend_cad,flags,sector_group,sector_code,sector_percent,country_code,country_percent
0,http://d-portal.org/q.html?aid=NL-KVK-27108436...,Oxfam Novib,NL-KVK-27108436,NL-KVK-27108436,WCLAC - Strengthening Resilience in E.J,onl_old,Closed,2016-05-01,2019-04-30,1094.0,...,0.0,400966.880,0.0,694602.800,0,"I.5.a. Government and civil society, general",Democratic participation and civil society,100.0,"Palestine, State of",100.0
1,http://d-portal.org/q.html?aid=NL-KVK-27108436...,Oxfam Novib,NL-KVK-27108436,NL-KVK-27108436,PMRS - Strengthening Resilience in E.J,onl_old,Closed,2016-05-01,2019-04-30,1094.0,...,0.0,361676.470,0.0,629696.940,0,"I.5.a. Government and civil society, general",Democratic participation and civil society,100.0,"Palestine, State of",100.0
2,http://d-portal.org/q.html?aid=NL-KVK-27108436...,Oxfam Novib,NL-KVK-27108436,NL-KVK-27108436,CO-ON - Strengthening Resilience in E.J,onl_old,Closed,2016-05-01,2019-04-30,1094.0,...,0.0,901313.750,0.0,1535724.500,0,"I.5.a. Government and civil society, general",Democratic participation and civil society,100.0,"Palestine, State of",100.0
3,http://d-portal.org/q.html?aid=NL-KVK-27108436...,Oxfam Novib,NL-KVK-27108436,NL-KVK-27108436,HO-HQ - Strengthening Resilience in E.J,onl_old,Closed,2016-05-01,2019-04-30,1094.0,...,0.0,24047.572,0.0,40050.688,0,"I.5.a. Government and civil society, general",Democratic participation and civil society,100.0,"Palestine, State of",100.0
4,http://d-portal.org/q.html?aid=NL-KVK-27108436...,Oxfam Novib,NL-KVK-27108436,NL-KVK-27108436,PARC - Strengthening Resilience in E.,onl_old,Closed,2016-05-01,2019-04-30,1094.0,...,0.0,212334.140,0.0,366335.720,0,"I.5.a. Government and civil society, general",Democratic participation and civil society,100.0,"Palestine, State of",100.0


## EDA - Sector and Aid Type

What columns can be used to determine the secotr / type of aid (e.g. food, cash, development, healthcare, etc)?

'reporting' column gives us the organization name 

In [6]:
#df_aid_activity_loc.reporting.unique()

'title' column gives us program names / short descriptions, but cannot be used as a standardized categorization of aid type

In [7]:
# pd.DataFrame(df_aid_activity_loc.title.unique()).to_string()

In [8]:
df_aid_activity_details.sector_group.unique()

array(['I.5.a. Government and civil society, general',
       'IV.2. Other Multisector', 'III.3.b. Tourism',
       'III.2.c. Construction', 'III.2.b. Mineral resources and mining',
       'III.2.a. Industry', 'III.1.c. Fishing', 'III.1.b. Forestry',
       'III.1.a. Agriculture', 'II.4. Banking and financial services',
       'II.3.d. Hybrid energy plants',
       'II.3.b. Energy generation, renewable sources',
       'II.2. Communications', 'II.1. Transport and storage',
       'I.6. Other social infrastructure and services',
       'I.4. Water supply and sanitation', 'I.2.a. Health, general',
       'I.1.a. Education, level nnspecified', nan,
       'VIII.2. Reconstruction and Rehabilitation',
       'XII. Unallocated / Unspecified',
       'III.3.a. Trade Policies and Regulations',
       'VIII.1. Emergency Response',
       'IV.1. General Environment Protection',
       'I.5.b. Conflict, peace and security',
       'I.1.c. Upper secondary education', 'I.2.b. Basic health',
       

In [9]:
violence_related_sector_groups = ['VIII.2. Reconstruction and Rehabilitation', 
                                    'VIII.1. Emergency Response', 
                                    'I.5.b. Conflict, peace and security', 
                                    'VIII.3. Disaster Preparedness',
                                    '920'
                                 ]

In [10]:
mask_violence_related_aid = df_aid_activity_details.sector_group.isin(violence_related_sector_groups)
df_aid_activity_details[mask_violence_related_aid].groupby('sector_group').sector_code.unique()

sector_group
920                                                       [Support to local and regional NGOs]
I.5.b. Conflict, peace and security          [Civilian peace-building, conflict prevention ...
VIII.1. Emergency Response                   [nan, Material relief assistance and services,...
VIII.2. Reconstruction and Rehabilitation    [Immediate post-emergency reconstruction and r...
VIII.3. Disaster Preparedness                [Multi-hazard response preparedness, Disaster ...
Name: sector_code, dtype: object

## Preprocessing

Isolate location and primary keys from location coverage

In [24]:
df_aid_activity_loc_ids = df_aid_activity_loc[['reporting', 'reporting_ref', 'slug', 'day_start', 'day_end', 'location_longitude', 'location_latitude']].drop_duplicates()
df_aid_activity_loc_ids.head()

,reporting,reporting_ref,slug,day_start,day_end,location_longitude,location_latitude
0,"Foreign Affairs, Trade and Development Canada ...",CA-3,gac-amc-4a,2018-03-28,2020-03-31,35.20329,31.92157
1,"Foreign Affairs, Trade and Development Canada ...",CA-3,gac-amc-4a,2018-11-07,2020-08-31,35.20329,31.92157
2,"Foreign Affairs, Trade and Development Canada ...",CA-3,gac-amc-4a,2018-10-05,2020-03-31,35.20329,31.92157
3,"Foreign Affairs, Trade and Development Canada ...",CA-3,gac-amc-4a,2019-03-27,2020-03-31,35.20329,31.92157
4,"Foreign Affairs, Trade and Development Canada ...",CA-3,gac-amc-4a,2019-03-28,2020-09-30,35.20329,31.92157


Isolate coverage details from non-location version of dataset

In [25]:
df_aid_activity_details_ids = df_aid_activity_details[['reporting', 'reporting_ref', 'slug', 'day_start', 'day_end', 'sector_group', 'sector_code', 'sector_percent']].drop_duplicates()
df_aid_activity_details_ids.head()

,reporting,reporting_ref,slug,day_start,day_end,sector_group,sector_code,sector_percent
0,Oxfam Novib,NL-KVK-27108436,onl_old,2016-05-01,2019-04-30,"I.5.a. Government and civil society, general",Democratic participation and civil society,100.00
6,Netherlands Enterprise Agency,NL-KVK-27378529,rvo-01,2009-01-01,2023-12-21,IV.2. Other Multisector,Food safety and quality,0.16
7,Netherlands Enterprise Agency,NL-KVK-27378529,rvo-01,2009-01-01,2023-12-21,IV.2. Other Multisector,Rural development,0.16
8,Netherlands Enterprise Agency,NL-KVK-27378529,rvo-01,2009-01-01,2023-12-21,IV.2. Other Multisector,Urban development,0.16
9,Netherlands Enterprise Agency,NL-KVK-27378529,rvo-01,2009-01-01,2023-12-21,IV.2. Other Multisector,Urban development and management,0.16


Join aid details to aid locations

In [26]:
df_aid_activity_full = df_aid_activity_details_ids.merge(df_aid_activity_loc_ids,
                                      how='left',
                                      on=['reporting', 'reporting_ref', 'slug', 'day_start', 'day_end']
                                     )

In [27]:
df_aid_activity_full.head(20)

,reporting,reporting_ref,slug,day_start,day_end,sector_group,sector_code,sector_percent,location_longitude,location_latitude
0,Oxfam Novib,NL-KVK-27108436,onl_old,2016-05-01,2019-04-30,"I.5.a. Government and civil society, general",Democratic participation and civil society,100.00,35.23245,31.7776
1,Oxfam Novib,NL-KVK-27108436,onl_old,2016-05-01,2019-04-30,"I.5.a. Government and civil society, general",Democratic participation and civil society,100.00,35.21371,31.7683
2,Netherlands Enterprise Agency,NL-KVK-27378529,rvo-01,2009-01-01,2023-12-21,IV.2. Other Multisector,Food safety and quality,0.16,NaN,NaN
3,Netherlands Enterprise Agency,NL-KVK-27378529,rvo-01,2009-01-01,2023-12-21,IV.2. Other Multisector,Rural development,0.16,NaN,NaN
4,Netherlands Enterprise Agency,NL-KVK-27378529,rvo-01,2009-01-01,2023-12-21,IV.2. Other Multisector,Urban development,0.16,NaN,NaN
5,Netherlands Enterprise Agency,NL-KVK-27378529,rvo-01,2009-01-01,2023-12-21,IV.2. Other Multisector,Urban development and management,0.16,NaN,NaN
6,Netherlands Enterprise Agency,NL-KVK-27378529,rvo-01,2009-01-01,2023-12-21,III.3.b. Tourism,Tourism policy and administrative management,0.79,NaN,NaN
7,Netherlands Enterprise Agency,NL-KVK-27378529,rvo-01,2009-01-01,2023-12-21,III.2.c. Construction,Construction policy and administrative management,0.16,NaN,NaN
8,Netherlands Enterprise Agency,NL-KVK-27378529,rvo-01,2009-01-01,2023-12-21,III.2.b. Mineral resources and mining,Fertilizer minerals,0.16,NaN,NaN
9,Netherlands Enterprise Agency,NL-KVK-27378529,rvo-01,2009-01-01,2023-12-21,III.2.b. Mineral resources and mining,Industrial minerals,0.47,NaN,NaN


match rate

In [28]:
df_aid_activity_full.location_latitude.isna().value_counts(normalize=True)

location_latitude
False    0.645721
True     0.354279
Name: proportion, dtype: float64

In [38]:
df_aid_activity_full[df_aid_activity_full.year_end>2025].location_latitude.isna().value_counts(normalize=True)

location_latitude
False    0.586384
True     0.413616
Name: proportion, dtype: float64

QA checks

In [23]:
# does the same aid (start and end date) map to two different locations? If so, I can't tell which location received which aid...

In [ ]:
mask_geo_ambiguous = df_aid_activity_full.
df_aid_activity_full.groupby('reporting', 'reporting_ref', 'slug', 'day_start', 'day_end')

In [34]:
# format date columns
df_aid_activity_full['day_start'] = pd.to_datetime(df_aid_activity_full['day_start'], errors='coerce')
df_aid_activity_full['day_end'] = pd.to_datetime(df_aid_activity_full['day_end'], errors='coerce')

# create date grouping columns
df_aid_activity_full['year_start'] = df_aid_activity_full['day_start'].dt.year
df_aid_activity_full['yearmo_start'] = df_aid_activity_full['day_start'].dt.to_period('M').astype(str)

df_aid_activity_full['year_end'] = df_aid_activity_full['day_end'].dt.year
df_aid_activity_full['yearmo_end'] = df_aid_activity_full['day_end'].dt.to_period('M').astype(str)

## Geoprocessing

In [ ]:
# convert event coordinates into spatial points
gdf_aid_activity = gpd.GeoDataFrame(
    df_aid_activity,
    geometry=gpd.points_from_xy(
        df_aid_activity['location_longitude'],
        df_aid_activity['location_latitude']
    ),
    crs=CRS
)

In [ ]:
# check and align CRSs
print("Events CRS:", gdf_aid_activity.crs)
print("Admin CRS:", gdf_boundaries.crs)
gdf_aid_activity = gdf_aid_activity.to_crs(gdf_boundaries.crs) # failsafe -- convert events CRS to admin boundaries CRS if not already aligned
assert(gdf_aid_activity.crs == gdf_boundaries.crs)
print('Confirmed CRSs are aligned')

In [ ]:
# perform spatial join (assign event coordinates to an administrative boundary)
gdf_aid_activity_joined = gpd.sjoin(
    gdf_aid_activity,
    gdf_boundaries[['adm1_name', 'adm1_pcode', 'adm2_name', 'adm2_pcode', 'geometry']],
    how='left',
    predicate='within'
)

In [ ]:
print("Total events:", len(gdf_aid_activity_joined))
print("Matched events:", gdf_aid_activity_joined['adm1_name'].notna().sum())
print("Unmatched events:", gdf_aid_activity_joined['adm1_name'].isna().sum())

In [ ]:
unmatched = gdf_aid_activity_joined[gdf_aid_activity_joined['adm1_name'].isna()]
if len(unmatched) > 0:
    display(unmatched.head())
else:
    print('All events matched successfully!')

In [32]:
gdf_aid_activity_joined.head()

,aid,reporting,reporting_ref,funder_ref,title,slug,status_code,day_start,day_end,day_length,...,location_longitude,location_latitude,location_precision,location_percent,geometry,index_right,adm1_name,adm1_pcode,adm2_name,adm2_pcode
0,http://d-portal.org/q.html?aid=CA-3-P002788005,"Foreign Affairs, Trade and Development Canada ...",CA-3,CA,Humanitarian Response to Populations at Risk o...,gac-amc-4a,Closed,2018-03-28,2020-03-31,734.0,...,35.20329,31.92157,2.0,NaN,POINT (35.20329 31.92157),6.0,West Bank,PS01,Ramallah,PS0130
1,http://d-portal.org/q.html?aid=CA-3-P006217001,"Foreign Affairs, Trade and Development Canada ...",CA-3,CA,United Nations Relief and Works Agency - Insti...,gac-amc-4a,Closed,2018-11-07,2020-08-31,663.0,...,35.20329,31.92157,2.0,NaN,POINT (35.20329 31.92157),6.0,West Bank,PS01,Ramallah,PS0130
2,http://d-portal.org/q.html?aid=CA-3-P006488001,"Foreign Affairs, Trade and Development Canada ...",CA-3,CA,Urgent Rehabilitation Services and Psycho-Soci...,gac-amc-4a,Closed,2018-10-05,2020-03-31,543.0,...,35.20329,31.92157,2.0,NaN,POINT (35.20329 31.92157),6.0,West Bank,PS01,Ramallah,PS0130
3,http://d-portal.org/q.html?aid=CA-3-P006585001,"Foreign Affairs, Trade and Development Canada ...",CA-3,CA,Humanitarian Response to Food Insecure Non-Ref...,gac-amc-4a,Closed,2019-03-27,2020-03-31,370.0,...,35.20329,31.92157,2.0,NaN,POINT (35.20329 31.92157),6.0,West Bank,PS01,Ramallah,PS0130
4,http://d-portal.org/q.html?aid=CA-3-P006585002,"Foreign Affairs, Trade and Development Canada ...",CA-3,CA,Humanitarian Response to Strengthen Food Secur...,gac-amc-4a,Closed,2019-03-28,2020-09-30,552.0,...,35.20329,31.92157,2.0,NaN,POINT (35.20329 31.92157),6.0,West Bank,PS01,Ramallah,PS0130


## Governorate Aggregates

### Active Org Count

In [ ]:
Keep in mind, this filtering step would theoretically be done in the applet UI (i.e. user filters for date range, and this code is run to produce the number of active orgs)
Additionally, I can run this code to create static reports / analyses

Conflict events are different than other risks. They are "sudden" (which is to say, they happen in a moment and then they end. This is slightly different than something like food insecurity. Aspects of violent conflict are similar -- there is a buildup, signs that things are getting worse,
the moment things escalate (i.e. violence happens), and then a de-escalation. But it happens more suddenly than the need for food, for money, for healthcare. I need to see this in parallel. Compare aspects of risk_conflict and risk_food_insecurity. Is risk_conflict that much different in 
terms of predictability, buildup, prevention, resolution?

Precursor to this: do orgs currently respond to violent conflicts? is there an aid type that addresses this such that I can even identify gaps?

Sprint 1:
    
    Questions we want to answer:
        - before the event was aid in place?
        - during the event, was there aid in place?
        - after the event, was there aid in place?

Beyond:
    Can we predict where/when future events will be? Can we mobilize aid accordingly?
        - for future event, will the aid be there before/during/after? Can we ensure it is ready before?


yearmo = '10/2026'

for each yearmo
    
    # programs active per yearmo
    active_programs = (title yearmo_start <= yearmo) AND (title yearmo_end >= yearmo) 
    
    # unique orgs among active programs
    active_orgs = active_programs.reporting.unique()

    # sector + population coverage (maybe breack pop cov into its own section if below gets too complicated)
    active_programs.groupby(sector).agg(
        size() # number of programs per sector
        reporting.nunique() # number of unique orgs per sector
        lamda x: sum(population covered by each program) # total population coverage
        lambda y: above lambda / total pop for that area = % pop served (caveat, assumes everyone in that area is in need instead of denominator being subset of people in population in need: Can I figure this out somehow? i.e. isolate people with given needs? eg are people with low income the ones receiving the program aid?)
        
        

### Sector Coverage

### Population Coverage

### Geographic Spread Coverage

The idea here is that we don't want to only prioritize populations by size, but we want to prevent populations from becoming concentrated in smaller areas (displacement -> geographic regression)

On map. highlight perimeter areas that are at risk of regression

Plan:

- figure out how to determine the type of aid (i.e. is this relevant to settler violence? to healthcare? food insecurity?)
- do I need to bring in the non-location dataset to do this?
- IMPORTANT: IS THERE ANY AID DESIGNATED TO CONFLICT-RELATED PREVENTION/RESOLUTION (ESP SETTLER VIOLENCE), IF NOT, GAP ANALYSIS CAN'T REALLY BE PERFORMED, BUT! CAN STILL SHOW CONFLICT AND SUGGEST AREAS FOR AID AND TYPES OF AID 
- aggregate per governorate (active org count, distinct sectors covered, % of population served or per capita coverage)
- build coverage score (org count, sector breadth, per capita coverage, etc) and normalize this on a 0-1 scale
- merge this into a master gdf dataset